In [ ]:
import fractions
import glob
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from collections import Counter, defaultdict

folder_path = 'D:\\results\\cft_KM_c50_2layers\\epochs'

In [ ]:
def parse_label_list(line):
    """Return list of labels from a tgt or pred line."""
    if 'None' in line or '[[' not in line or ']]' not in line:
        return []
    start = line.find("[[")
    end   = line.find("]]") + 2
    list_str = line[start:end].replace("Fraction", "fractions.Fraction")
    try:
        pairs = eval(list_str, {"fractions": fractions})
        return [lbl for lbl, _ in pairs]
    except Exception:
        return []


def analyze_missing_labels(folder, order_insensitive=True):
    file_list = sorted(
        glob.glob(f"{folder}/eval.valid.cfts.*"),
        key=lambda x: int(x.split('.')[-1])
    )

    missing_counts = Counter()
    replacement_counts = defaultdict(Counter)
    correct_counts = Counter()

    for f in file_list[-1:]:
        with open(f) as fh:
            tgt_labels = []
            for ln in fh:
                if ln.startswith("tgt"):
                    tgt_labels = parse_label_list(ln)

                elif ln and ln[0].isdigit():
                    pred_labels = parse_label_list(ln)
                    if not tgt_labels:
                        continue

                    if not order_insensitive:
                        max_len = max(len(tgt_labels), len(pred_labels))
                        for i in range(max_len):
                            tgt_lbl  = tgt_labels[i]  if i < len(tgt_labels)  else None
                            pred_lbl = pred_labels[i] if i < len(pred_labels) else None
                            if tgt_lbl is None:
                                continue
                            if pred_lbl == tgt_lbl:
                                correct_counts[tgt_lbl] += 1
                            else:
                                missing_counts[tgt_lbl] += 1
                                replacement_counts[tgt_lbl][pred_lbl if pred_lbl is not None else "(empty)"] += 1
                        continue

                    # Permutation-invariant (multiset-based)
                    tgt_ctr  = Counter(tgt_labels)
                    pred_ctr = Counter(pred_labels)

                    overlap = {}
                    for lbl, ct in tgt_ctr.items():
                        m = min(ct, pred_ctr.get(lbl, 0))
                        if m > 0:
                            overlap[lbl] = m
                            correct_counts[lbl] += m

                    tgt_rem  = tgt_ctr.copy()
                    pred_rem = pred_ctr.copy()
                    for lbl, m in overlap.items():
                        tgt_rem[lbl]  -= m
                        pred_rem[lbl] -= m
                        if tgt_rem[lbl]  <= 0: del tgt_rem[lbl]
                        if pred_rem.get(lbl, 0) <= 0: pred_rem.pop(lbl, None)

                    tgt_unmatched  = [lbl for lbl in sorted(tgt_rem)  for _ in range(tgt_rem[lbl])]
                    pred_unmatched = [lbl for lbl in sorted(pred_rem) for _ in range(pred_rem[lbl])]

                    k = min(len(tgt_unmatched), len(pred_unmatched))
                    for i in range(k):
                        missing_counts[tgt_unmatched[i]] += 1
                        replacement_counts[tgt_unmatched[i]][pred_unmatched[i]] += 1

                    for i in range(k, len(tgt_unmatched)):
                        missing_counts[tgt_unmatched[i]] += 1
                        replacement_counts[tgt_unmatched[i]]["(empty)"] += 1

    return missing_counts, replacement_counts, correct_counts

In [ ]:
missing_counts, replacement_counts, correct_counts = analyze_missing_labels(
    folder_path, order_insensitive=True
)

desired_order = [
    "su2",
    "su3", "sp4", "g2",
    "su4", "so7", "sp6",
    "su5", "so8", "so9", "sp8", "f4",
    "su6", "so10", "so11",
    "so12", "e6",
    "e7",
    "e8"
]

raw_true = set(missing_counts) | set(replacement_counts) | set(correct_counts)
labels = [lbl for lbl in desired_order if lbl in raw_true]
labels += sorted(raw_true - set(labels))

raw_pred = {p for repls in replacement_counts.values() for p in repls} | set(correct_counts)
all_replacements = [lbl for lbl in desired_order if lbl in raw_pred]
all_replacements += sorted(raw_pred - set(all_replacements))
if "(empty)" in raw_pred and "(empty)" not in all_replacements:
    all_replacements.append("(empty)")
pred_index = {lbl: j for j, lbl in enumerate(all_replacements)}

# Build raw count confusion matrix
conf_mat = np.zeros((len(labels), len(all_replacements)), dtype=int)
for i, true_lbl in enumerate(labels):
    if true_lbl in pred_index:
        conf_mat[i, pred_index[true_lbl]] += correct_counts.get(true_lbl, 0)
    for pred_lbl, ct in replacement_counts.get(true_lbl, {}).items():
        j = pred_index.get(pred_lbl)
        if j is not None:
            conf_mat[i, j] += ct

n_rows, n_cols = conf_mat.shape
diag_len = min(n_rows, n_cols)
diag_mask    = np.zeros((n_rows, n_cols), dtype=bool)
diag_mask[np.arange(diag_len), np.arange(diag_len)] = True
offdiag_mask = ~diag_mask

# Row-normalize off-diagonal counts only
conf_off = conf_mat.astype(float)
conf_off[diag_mask] = 0.0
row_sums_off    = conf_off.sum(axis=1, keepdims=True)
rows_with_errors = (row_sums_off[:, 0] > 0)
conf_off_percent = np.zeros_like(conf_off)
conf_off_percent[rows_with_errors] = (
    conf_off[rows_with_errors] / row_sums_off[rows_with_errors]
) * 100.0

# Epsilon smoothing for log-scale visualization
eps = 1e-3
conf_smooth = conf_off_percent + eps * offdiag_mask
off_sums = (conf_smooth * offdiag_mask).sum(axis=1, keepdims=True)
conf_smooth[rows_with_errors] = conf_smooth[rows_with_errors] / off_sums[rows_with_errors]
conf_smooth[diag_mask] = np.nan

cmap = plt.cm.viridis.copy()
cmap.set_bad(color="white")
norm = mcolors.LogNorm(vmin=eps, vmax=1.0)

plt.figure(figsize=(8, 6))
im = plt.imshow(conf_smooth, aspect="auto", cmap=cmap, norm=norm)

plt.xticks(ticks=np.arange(len(all_replacements)), labels=all_replacements, rotation=45, fontsize=13)
plt.yticks(ticks=np.arange(len(labels)), labels=labels, fontsize=13)
plt.xlabel("Predicted Label", fontsize=16)
plt.ylabel("True Label", fontsize=16)
plt.title(r"Confusion Matrix for $50 \leq c^{(T)} \leq 100$", fontsize=16)

for i in range(n_rows):
    for j in range(n_cols):
        if diag_mask[i, j] or conf_mat[i, j] == 0:
            continue
        plt.text(j, i, f"{conf_mat[i, j]}", ha="center", va="center",
                 fontsize=10, color="white", alpha=0.85)

cbar = plt.colorbar(im)
cbar.set_label("Per-row Normalized Frequency", fontsize=16)
cbar.ax.tick_params(labelsize=18)

plt.tight_layout()
plt.savefig("confusion_c50-100.pdf", dpi=300)
plt.show()